# Configure

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Packages
import requests
import os
import re
from os.path import join
from pathlib import Path
import yaml
from yaml.loader import SafeLoader
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import numpy as np
import rioxarray as rio

import unsafe.download as undown
import unsafe.files as unfile
import unsafe.unzip as ununzip
import unsafe.exp as unexp
import unsafe.ddfs as unddf
import unsafe.ensemble as unens

In [3]:
# Name the fips, statefips, stateabbr, and nation that
# we are using for this analysis
# We pass these in as a list even though the framework currently
# processes a single county so that it can facilitate that
# expansion in the future
# TODO - could make sense to define these in the future
# in json or other formats instead of as input in code
fips_args = {
    'FIPS': ['42101'], 
    'STATEFIPS': ['42'],
    'STATEABBR': ['PA'],
    'NATION': ['US']
}
FIPS = fips_args['FIPS'][0]
NATION = fips_args['NATION'][0]

In [4]:
# We need to pass in a config file that sets up
# constants and the structure for downlading data
# For the directory structure of our case study, 
# we use the following 
ABS_DIR = Path().absolute().parents[0]

CONFIG_FILEP = join(ABS_DIR, 'config', 'config.yaml')
# Open the config file and load
with open(CONFIG_FILEP) as f:
    CONFIG = yaml.load(f, Loader=SafeLoader)

# Wildcards for urls
URL_WILDCARDS = CONFIG['url_wildcards']

# Get the file extensions for api endpoints
API_EXT = CONFIG['api_ext']

# Get the CRS constants
NSI_CRS = CONFIG['nsi_crs']

# Dictionary of ref_names
REF_NAMES_DICT = CONFIG['ref_names']

# Dictionary of ref_id_names
REF_ID_NAMES_DICT = CONFIG['ref_id_names']

# Coefficient of variation
# for structure values
COEF_VARIATION = CONFIG['coef_var']

# First floor elevation dictionary
FFE_DICT = CONFIG['ffe_dict']

# Number of states of the world
N_SOW = CONFIG['sows']

# Data for flood depth grids
# Get hazard model variables
HAZ_FILEN = CONFIG['haz_filename']
# Get CRS for depth grids
HAZ_CRS = CONFIG['haz_crs']
# Ensemble members
HAZ_NENS = CONFIG['haz_nens']
# Number of columns for each depth grid
HAZ_NCOLS = CONFIG['haz_ncols']
# Num rows for each depth grid
HAZ_NROWS = CONFIG['haz_nrows']
# Lower left x coordinate
HAZ_XLL = CONFIG['haz_xll']
# Lower left y coordinate
HAZ_YLL = CONFIG['haz_yll']
# Cell resolution
HAZ_RES = CONFIG['haz_res']
# NODATA values
HAZ_NODATA = CONFIG['haz_nodata']

# Get the files we need downloaded
DOWNLOAD = pd.json_normalize(CONFIG['download'], sep='_').T

# We can also specify the filepath to the
# raw data directory
FR = join(ABS_DIR, "data", "raw")

# And external - where our hazard data should be
FE = join(FR, "external")

# Set up interim and results directories as well
# We already use "FR" for raw, we use "FO" 
# because you can also think of results
# as output
FI = join(ABS_DIR, "data", "interim")
FO = join(ABS_DIR, "data", "results")

# "Raw" data directories for exposure, vulnerability (vuln) and
# administrative reference files
EXP_DIR_R = join(FR, "exp")
VULN_DIR_R = join(FR, "vuln")
REF_DIR_R = join(FR, "ref")
# Haz is for depth grids
HAZ_DIR_R = join(FE, "haz")
# Pol is for NFHL
POL_DIR_R = join(FR, "pol")

# Unzip directory 
UNZIP_DIR = join(FR, "unzipped")

# We want to process unzipped data and move it
# to the interim directory where we keep
# processed data
# Get the filepaths for unzipped data
# We unzipped the depth grids (haz) and 
# ddfs (vuln) into the "external"/ subdirectory
HAZ_DIR_UZ = join(UNZIP_DIR, "external", "haz")
POL_DIR_UZ = join(UNZIP_DIR, "pol")
REF_DIR_UZ = join(UNZIP_DIR, "ref")
VULN_DIR_UZ = join(UNZIP_DIR, "external", "vuln")

# "Interim" data directories
EXP_DIR_I = join(FI, "exp")
VULN_DIR_I = join(FI, "vuln")
REF_DIR_I = join(FI, "ref")
# Haz is for depth grids
HAZ_DIR_I = join(FI, "haz")
# Pol is for NFHL
POL_DIR_I = join(FI, "pol")

# Download and unzip data

In [5]:
wcard_dict = {x: fips_args[x[1:-1]][0] for x in URL_WILDCARDS}
undown.download_raw(DOWNLOAD, wcard_dict,
                    FR, API_EXT)

Downloaded from: https://nsi.sec.usace.army.mil/nsiapi/structures?fips=42101
Downloaded from: https://phl.carto.com/api/v2/sql?filename=opa_properties_public&format=geojson&skipfields=cartodb_id&q=SELECT+*+FROM+opa_properties_public
Downloaded from: https://opendata.arcgis.com/api/v3/datasets/ab9e89e1273f445bb265846c90b38a96_0/downloads/data?format=geojson&spatialRefId=4326&where=1%3D1
Downloaded from: https://opendata.arcgis.com/api/v3/datasets/84baed491de44f539889f2af178ad85c_0/downloads/data?format=geojson&spatialRefId=4326&where=1%3D1
Downloaded from: https://hazards.fema.gov/nfhlv2/output/County/420757_20230701.zip
Downloaded from: https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_42_tract.zip
Downloaded from: https://www2.census.gov/geo/tiger/TIGER2022/BG/tl_2022_42_bg.zip
Downloaded from: https://www2.census.gov/geo/tiger/TIGER2022/TABBLOCK20/tl_2022_42_tabblock20.zip
Downloaded from: https://static-data-screeningtool.geoplatform.gov/data-versions/1.0/data/score/download

In [6]:
ununzip.unzip_raw(FR, UNZIP_DIR)

Unzipped: nfhl
Unzipped: zcta
Unzipped: county
Unzipped: bg
Unzipped: tract
Unzipped: block
Unzipped: ddfs
Unzipped: RIFT_domain
Unzipped: Irene


# Prepare data for ensemble

The study domain corresponds to 12 digit USGS hydrological unit code (HUC) watershed 020402031008. We will spatially merge the NSI structures and Philadelphia data to this extent. We will restrict the other downloaded geospatial data to objects that intersect with this (e.g., Census Tracts that overlap). We may clip these for plotting purposes later.

## Study area boundary

In [5]:
CLIP_SHP_FILEP = join(HAZ_DIR_UZ, 'RIFT_domain', 'domain_1.shp')
clip_geo = gpd.read_file(CLIP_SHP_FILEP)

/Users/f006dwr/miniforge3/envs/nsi_fit/lib/python3.12/site-packages/pyogrio/geopandas.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  res = pd.to_datetime(ser, **datetime_kwargs)


## Process exposure

We will start by subsetting several datasets to the envelope of our clip polygon and then do the processing on those subsets.

### Get subsets of NSI and Philly data

In [6]:
# Load in the NSI and Philly assessor, parcel, and footprint data
nsi_gdf = unexp.get_nsi_geo(FIPS, NSI_CRS, EXP_DIR_R)

assess_cols = ['assessment_date', 'basements', 'building_code',
               'building_code_description', 'building_code_description_new',
               'category_code', 'category_code_description', 'census tract',
               'exterior_condition', 'garage_type', 'general_construction',
               'interior_condition','location', 'market_value',
               'market_value_date', 'number_stories', 'owner_1',
               'parcel_number', 'sale_date', 'sale_price',
               'quality_grade', 'taxable_building', 'exempt_building',
               'total_area', 'total_livable_area',
               'topography', 'unit', 'year_built',
               'other_building', 'garage_type',
               'year_built_estimate', 'zoning']
assess = gpd.read_file(join(EXP_DIR_R, FIPS, 'assess.geojson'),
                       mask=clip_geo, columns=assess_cols)

parcel = gpd.read_file(join(EXP_DIR_R, FIPS, 'parcel.geojson'),
                       mask=clip_geo)
bld_fp = gpd.read_file(join(EXP_DIR_R, FIPS, 'bldfp.geojson'),
                       mask=clip_geo)

Prepared geodataframe


/Users/f006dwr/miniforge3/envs/nsi_fit/lib/python3.12/site-packages/pyogrio/geopandas.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  res = pd.to_datetime(ser, **datetime_kwargs)
/Users/f006dwr/miniforge3/envs/nsi_fit/lib/python3.12/site-packages/pyogrio/geopandas.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  res = pd.to_datetime(ser, **datetime_kwargs)


#### NSI subset

We'll follow existing UNSAFE functions to get our NSI dataset. We're going to include any residential structure in occupancy type RES1 and RES3

In [7]:
# Set the values that we pass into the get_struct_subset function
occtype_list=['RES1-1SNB', 'RES1-2SNB', 'RES1-1SWB', 'RES1-2SWB',
              'RES1-SLNB', 'RES1-SLWB', 'RES1-3SNB', 'RES1-3SWB',
              'RES3A', 'RES3B', 'RES3C', 'RES3D', 'RES3E', 'RES3F']
sub_string = 'occtype.isin(@occtype_list)'
nsi_filt = unexp.get_struct_subset(nsi_gdf,
                                   filter=sub_string,
                                   occtype_list=occtype_list)

EXP_OUT_FILEP = join(EXP_DIR_I, FIPS, 'nsi_res.pqt')
unfile.prepare_saving(EXP_OUT_FILEP)

# Clip to our boundary to reduce file size
nsi_clip_out = gpd.clip(nsi_filt, clip_geo.to_crs(nsi_filt.crs))

# Write file
nsi_clip_out.to_parquet(EXP_OUT_FILEP, index=False)

# Helpful summaries
print('Total NSI structures: {}'.format(len(nsi_gdf)))
print('Total NSI res structures: {}'.format(len(nsi_filt)))
print('Total NSI res structures in study area: {}'.format(len(nsi_clip_out)))

Total NSI structures: 527752
Total NSI res structures: 476174
Total NSI res structures in study area: 96630


#### Philly data subsets

We want to use the assessment data to identify residential structures. Then we will subset the building footprints and parcels correspondingly. To match up records, we will link `assess['parcel_number']` to `parcel['BRT_ID']` to `bld_fp['PARCEL_ID_NUM']`.

Condos will require extra processing. From the Maps@Phila.gov email: “The buildings are matched via their centroid to the PWD Parcels for their parcelid, they could use the parcelid to connect to PWD Parcels, then use the BRT_ID field in the PWD Parcels to get to the OPA Tax Accounts.  This won’t be the cleanest solution for condos, but there’s no real system for handling those anywhere.  You can tell [redacted] she’s welcome to point out any mismatches she finds directly to me, I’ve worked with her before on other projects.” We describe the condo processing approach above the corresponding cell block.

We start by processing the assessor data. We use the `building_code_description` column to identify RES1 and RES3 mappings by sampling records and checking the properties in street view apps (Google and Philadelphia's own) and Philadelphia Properties/Atlas apps. Some building code descriptions appear to uniformly map to RES1 or RES3, but some are mixed. For example, some buildings are coded as twin row homes, which we consider RES3, but their neighbor was demolished so effectively the property is RES1. For our 'main' sample, we use building footprint processing (to identify detached footprints) and sq. ft. statistics on individual row homes to identify likely RES1. For sensitivity checks, we use majority mappings for building code descriptions to occupancy type (and a few other checks). 

Below we split the `building_code_description` column in a way that gives us reduced form information for a subset of structure types we can look through manually. 

In [8]:
def split_bld_code(bld_desc):

    """
    Split a building code description into tokens based on the first numeric value.

    This function takes a building code description and splits it into tokens where
    all text before the first numeric value becomes one token, and all subsequent
    words (including numeric values) become individual tokens.

    Parameters
    ----------
    bld_desc : str
        A string containing the building code description.
        Example: 'APT 2-4 UNITS 3.5 STY MAS'

    Returns
    -------
    list
        A list where the first element is all text before the first number (as one string),
        followed by all remaining words as individual elements.
        Example: ['APT', '2-4', 'UNITS', '3.5', 'STY', 'MAS']
        If no numeric values are found, returns the entire description as a single element list.

    Examples
    --------
    >>> split_bld_code('APT 2-4 UNITS 3.5 STY MAS')
    ['APT', '2-4', 'UNITS', '3.5', 'STY', 'MAS']
    
    >>> split_bld_code('DET W/GAR 2 STY MASONRY')
    ['DET W/GAR', '2', 'STY', 'MASONRY']
    """

    if bld_desc is None:
        return bld_desc

    # Split the building code description into words
    full_code = bld_desc.split()

    # Find the index of the first string with a number as first character
    first_num_idx = next((i for i, word in enumerate(full_code) if word[0].isdigit()), None)
    
    if first_num_idx is not None:
        # Join everything before the first number as one token
        prefix = ' '.join(full_code[:first_num_idx])
        # Keep remaining words as separate tokens
        remaining = full_code[first_num_idx:]
        return [prefix] + remaining
    else:
        return [' '.join(full_code)]

In [9]:
print('Total tax records in study area: {}'.format(len(assess)))

# We want to retain structures with a building code description
assess_sub = assess[assess['building_code_description'].notnull()].copy()
# split up the building code description field
assess_sub.loc[:, 'bld_code_split'] = assess_sub['building_code_description'].apply(split_bld_code)

# get the occupancy type code and the remaining token 
# into separate columns
assess_sub.loc[:, 'bld_type'] = assess_sub['bld_code_split'].apply(lambda x: x[0])
assess_sub.loc[:, 'bld_code_rest'] = assess_sub['bld_code_split'].apply(lambda x: x[1:])
# helpful to have the rest as a single string for some inspections
# can drop the last token though (usually foundation type)
assess_sub.loc[:, 'bld_code_rest_str'] = assess_sub['bld_code_rest'].apply(lambda x: ' '.join(x[:-1]))

# We want to subset to the category codes that may have res buildings
cat_codes = ['1', '2', '3', '14']
assess_sub = assess_sub[assess_sub['category_code'].str.strip().isin(cat_codes)]

# We do not want "VACANT" 
assess_sub = assess_sub[~assess_sub['bld_type'].str.contains('VACANT')]

# We can also drop anything with empty bld_code_rest_str
assess_non_res = assess_sub[assess_sub['bld_code_rest_str'] == '']
assess_sub = assess_sub[assess_sub['bld_code_rest_str'] != '']

print('Sample of tax records in study area: {}'.format(len(assess_sub)))

Total tax records in study area: 123586
Sample of tax records in study area: 106482


Below, we take the reduced form building codes to sample 10 properties (or the number of properties in the new code) for manual checking. We generated two of these files to allow for two analysts to check each others mappings and converge on processing rules for main and sensitivity analyses. We comment out the sample writing lines to avoid overwriting data generated in our analysis. The files we generated and coded are available for others to inspect. They may also generate new samples (change the file suffix). 

In [10]:
# sample a few records from each bld_type group
samples = assess_sub.groupby('bld_type').apply(lambda x: x.sample(n=min(10, len(x)))).reset_index(drop=True)
# write out the parcel numbers and a few other columns and start 
# checking the ddf pairing
check_cols = ['parcel_number', 'bld_type', 'bld_code_rest_str',
              'building_code', 'category_code_description', 'zoning']
# check_dir = join(EXP_DIR_I, 'check_records')
# file_suf = '020425.csv'
# check_filep = join(check_dir, 'check_codes_' + file_suf)
# unfile.prepare_saving(check_filep)
# samples[check_cols].to_csv(check_filep, index=False)

/var/folders/d2/g0h08s551zb2hz_ws2g4ggh400hbd0/T/ipykernel_57034/2605892050.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  samples = assess_sub.groupby('bld_type').apply(lambda x: x.sample(n=min(10, len(x)))).reset_index(drop=True)


We don't want to use the parcel centroid as a way to link with the flood hazard. We want to use the building footprint. We have to link the assessor records to parcels and building footprints. We need the parcels dataset because that's how we can merge the building footprints in. 

First, we will drop the `bld_type` that we identified as not having any residential structures. The remaining records are our residential subset. 

In [11]:
# Identified manually by evaluating partial (but sometimes full) samples of unique bld_type
drop_bld_codes = ['HOTEL', 'PRIV GAR']
assess_res = assess_sub[~assess_sub['bld_type'].isin(drop_bld_codes)].copy()

print('Sample of res tax records in study area: {}'.format(len(assess_res)))

Sample of res tax records in study area: 106300


We also want to add the taxable and exempt building value for our structure value

In [12]:
assess_res['val_struct'] = assess_res['taxable_building'] + assess_res['exempt_building']

We should also subset based on acceptable exterior and interior condition
The [documentation](https://metadata.phila.gov/#home/datasetdetails/5543865f20583086178c4ee5/representationdetails/55d624fdad35c7e854cb21a4/?view_287_per_page=100&view_287_page=1) tells us for exterior condition:

7. VACANT – No occupancy. FHA, VA, FNMA signs may be on the property. Property has been secured with fresh plywood over doors and windows.
8. SEALED – Doors and windows have been covered over by plywood, tin, concrete block or stucco. No interior access.
9. STRUCTURALLY COMPROMISED, OPEN TO THE WEATHER - Some or no windows, no door or door open, evidence of past abuse by vandals such as graffiti, missing railings, deteriorated wood and metal, etc. Scorch marks and/or fire and water damage to exterior brick, siding, bays, etc. Broken windows with blackened and charred interior.

For interior: 

6. Vacant – No occupancy. FHA, VA, FNMA signs may be on the property.
Property has been secured with fresh plywood over doors and windows.
7. Sealed / Structurally Compromised, Open to the Weather –
Doors and windows have been covered over by plywood, tin, concrete block or
stucco. No interior access. Some or no windows, no door or door open, evidence
of past abuse by vandals such as graffiti, missing railings, deteriorated wood and
metal, etc. Scorch marks and/or fire and water damage to exterior brick, siding,
bays, etc. Broken windows with blackened and charred interior.


In [1012]:
assess_res = assess_res[(~assess_res['interior_condition'].str.strip().isin(['6', '7'])) &
                        (~assess_res['exterior_condition'].str.strip().isin(['7', '8', '9']))].copy()
print('Sample of occupied res tax records in study area: {}'.format(len(assess_res)))
print('Unique res addresses in study area: {}'.format(len(assess_res['location'].unique())))

Sample of occupied res tax records in study area: 103995
Unique res addresses in study area: 95024


These are helpful numbers to keep in mind. While they may seem like lower and upper bounds, I don't think the are. We know there are tax records that correspond to the same structure, so we expect to have less than 103995. But we also have records that correspond to multiple structures (e.g., apartment complexes), so that will tend to drive the final sample up. However, there are also tax records that have erroneous parcel numbers (do not link to a BRT_ID in the parcel data) and/or tax records that correspond to buildings that are no longer there (or sometimes new buildings that don't have a building footprint ID yet). 

These are benchmark numbers. If we come in around 100k, I think our processing did a good job. We should of course see what tax records are not represented in our final dataset and quantify *why* that's the case, but if we come in around 100k I think it's a good benchmark. Btw, NSI comes in at around 96k, so similar benchmark.  From 104k, data error in one of tax/parcle/bld_fp can drive the number down, as well as aggregation of records to single structures. But I am expecting a decent number of disaggregating a record to many building footprints. From my checks so far, I expect more aggregation than disaggregation (e.g., there are some condos with a hundred units but I haven't seen a single apartment complex with 100 buildings) so I could see us being closer to 95k than 104k, especially considering the data error as well. 

Most of the assessment records get matched to building footprints successfully but there are a few inconsistencies because of the parcel boundaries not overlapping with the building enough for the centroid method to work. There are a number of records matched to multiple footprints (the entire footprint) in a way that makes it ambiguous to know the structure footprint for the record. This is an experimental section to see if we can get the residential structure for each record better than we can using the existing linkages. This will include some assumption-driven processing about how to filter for garages and other appurtenant structures that will have analogues in more of a post-processing step for the existing linked data. Will probably treat this as an experimental notebook that demonstrates the results of the comparisons since for clarity in the main analysis we'll want to have the assumptions baked in for more readability.   

In [928]:
# Start by overlaying the bld_fp with parcels
# Most records have direct links to assess_res through parcel_number/BRT_ID
# These are the ones we want to overlay - we will do links for
# nonmatched afterwards 

par_cols = ['BRT_ID', 'PARCEL_ID', 'ADDRESS', 'geometry']
tax_cols = ['parcel_number', 'bld_type', 'bld_code_rest',
            'building_code_description_new',
            'val_struct',
            'basements', 'unit']

assess_linked = assess_res[assess_res['has_parcel_match']].copy()
assess_no_link = assess_res[~assess_res['has_parcel_match']].copy()

direct_matches = parcel[par_cols].merge(
    assess_linked[tax_cols],
    right_on='parcel_number',
    left_on='BRT_ID'
)

print('Direct tax-parcel matches: {}'.format(len(direct_matches)))

# We can also create indirect matches by limiting parcels
# to those not in direct_matches (based on BRT_ID)
# and then doing a spatial join with assess_no_link
# we want to only keep the first of entries with
# duplicate ids
parcel_no_match = parcel[~parcel['BRT_ID'].isin(direct_matches['BRT_ID'])]
indirect_matches = gpd.sjoin(parcel_no_match[par_cols],
                             assess_no_link[tax_cols + ['geometry']],
                             predicate='contains',
                             how='inner')
print('Tax-parcel matches from sp joins: {}'.format(len(indirect_matches)))
indirect_matches['geometry'] = indirect_matches['geometry'].normalize()
indirect_matches = indirect_matches.drop_duplicates(subset='geometry', keep='first')
print('"unique" tax-parcel matches post drop duplicates: {}'.format(len(indirect_matches)))

tax_pc_matches = pd.concat([direct_matches, indirect_matches], axis=0)

# tax records are uniquely linked to parcels unless
# they refer to condos/apts, in which case we only want
# to bring the remainder of those tax records in later for aggregating
# things like structure value and then dividing across
# building footprints on the parcel
# in cases where this is only one building footprint, 
# you'd just keep what you aggregated
pc_res_dir = gpd.GeoDataFrame(tax_pc_matches,
                              geometry=tax_pc_matches['geometry'],
                              crs=parcel.crs)

# Because we do an overlay with bld_fp, there are touching buildings
# treated as different bld_fp_o even though they overlap with the 
# same parcel. We want to get the unary union of these touching
# building footprints because for our purposes the
# spatial precision comes from any unique built structures
# located at a specific parcel
bld_fp_diss = temp = bld_fp.dissolve().explode()
bld_fp_o = gpd.overlay(pc_res_dir, bld_fp_diss[['geometry']], how='intersection')

# Convert the new footprints to epsg 5070 for area calculations
bld_fp_o['m2_bld'] = bld_fp_o.to_crs(epsg='5070').area

# Drop links where area threshold not met
bld_fp_min_m2 = 10
bld_fp_drop = bld_fp_o.loc[bld_fp_o['m2_bld'] <= bld_fp_min_m2]
bld_fp_o = bld_fp_o.loc[bld_fp_o['m2_bld'] > bld_fp_min_m2]

# Bring back links where area threshold was not met
# but it's the only building reference available for the
# parcel
merge_back = pc_res_dir[~pc_res_dir['parcel_number'].isin(bld_fp_o['parcel_number'])]['parcel_number']
merge_back_bld = bld_fp_drop[bld_fp_drop['parcel_number'].isin(merge_back)]
bld_fp_o = pd.concat([bld_fp_o, merge_back_bld], axis=0)

# print out the number of unmatched parcels
unmatched = len(pc_res_dir[~pc_res_dir['parcel_number'].isin(bld_fp_o['parcel_number'])])
print('Unmatched tax-bld_fp in study area: {}'.format(unmatched))
matched = len(pc_res_dir[pc_res_dir['parcel_number'].isin(bld_fp_o['parcel_number'])])
print('Matched tax-bld_fp in study area: {}'.format(matched))
match_prop = (matched)/len(pc_res_dir)
print('Proportion of matched tax records in study area: {}'.format(match_prop))

Direct tax-parcel matches: 94273
Tax-parcel matches from sp joins: 1300
"unique" tax-parcel matches post drop duplicates: 98
Unmatched tax-bld_fp in study area: 260
Matched tax-bld_fp in study area: 94111
Proportion of matched tax records in study area: 0.997244916340825


In [929]:
# Get a new id
# Records with identical geometry should have same building footprint id
# Because of direct_matches above, we will only have 1 record per
# group but this is a more generalizable solution than other options
bld_fp_o['geometry'] = bld_fp_o['geometry'].normalize()
bld_fp_o['bfid'] = bld_fp_o.groupby('geometry').ngroup()

# Calculate the number of parcels linked to this building footprint
bld_fp_o['n_parcels'] = bld_fp_o.groupby('bfid')['parcel_number'].transform('nunique')
# and vice versa
bld_fp_o['n_bld'] = bld_fp_o.groupby('parcel_number')['bfid'].transform('nunique')

Any building linked to one parcel gets assigned to that parcel.

For remaining cases, will require some combination of disaggregation of tax record to buildings and aggregating info from tax records before disaggregating to buildings. Note that some condos have 1:1 (i.e., no disaggregation required) and still need to be linked to unmatched tax records for aggregation. 

In [930]:
# pc_bld will be our final dataframe of links
# we'll append processed subset dfs into a list
# and then concat into pc_bld when done
pc_bld_l = []

# Separate parcels with one building from more complex cases
# Add our simple cases to our processed dfs list
par_one_bld_match = bld_fp_o[bld_fp_o['n_bld'] == 1]
pc_bld_l.append(par_one_bld_match)
par_one_bld_many = bld_fp_o[bld_fp_o['n_bld'] > 1]

print('Number of 1 to 1 matches: {}'.format(len(par_one_bld_match)))
print('Number of 1 to many matches: {}'.format(len(par_one_bld_many['parcel_number'].unique())))

Number of 1 to 1 matches: 91527
Number of 1 to many matches: 2584


For the one parcel to many bld cases, we'll start by dropping any footprints that are a small proportion of the max building footprint associated with the parcel. We can sometimes see complexes with a building 2 to 3 times larger than others, maybe even a bit more, but it's very uncommon to have one structure much larger than the others. They tend to be a similar size anyway. So, we will drop footprints that are a small proportion. There will still be some cases that are ambiguous after this. We can split on bld_type for apt/condo vs. other structures because we can try to be a bit more restrictive with the area ratio for the latter and can do another round of checking. Also, we are more comfortable assuming the largest area structure is the main building for these, whereas for other apt/condo we are more comfortable assuming we need to disaggregate across the structures. 

In [931]:
par_one_bld_many['area_ratio'] = (par_one_bld_many['m2_bld'] / 
                                  par_one_bld_many.groupby('parcel_number')['m2_bld'].transform('max').copy())

# This filter is arbitrary but QA checks suggests effective
# First, it has a rather high threshold for the different sizes of
# footprints linked to a parcel. This is a good filter to have for
# the structures we want to drop like garages, but there are some huge 
# complexes that have a lot of adjacent buildings or huge units
# and a few detached units that end up taking a small proportion. That's
# where the 100 m2 threshold comes in
pc_one_many_keep = par_one_bld_many[(par_one_bld_many['area_ratio'] >= .95) |
                                    (par_one_bld_many['m2_bld'] >= 100)].copy()

# Calculate bld linked to each parcel
pc_one_many_keep['n_bld'] = pc_one_many_keep.groupby('parcel_number')['bfid'].transform('nunique').copy()

# Separate parcels with one building from more complex cases
# Add our simple cases to our processed dfs list
pc_one_bld_lower_conf = pc_one_many_keep[pc_one_many_keep['n_bld'] == 1].copy()
pc_bld_l.append(pc_one_bld_lower_conf)
# Combo of pc to disagg and bld_type we'd like to 
# do a bit more processing on to filter out garages/similar
pc_one_bld_many = pc_one_many_keep[pc_one_many_keep['n_bld'] > 1].copy()

print('Lower confidence 1 to 1 matches: {}'.format(len(pc_one_bld_lower_conf)))
print('Remaining 1 to many matches: {}'.format(len(pc_one_bld_many['parcel_number'].unique())))
print('Remaining buildings: {}'.format(len(pc_one_bld_many)))

Lower confidence 1 to 1 matches: 2346
Remaining 1 to many matches: 238
Remaining buildings: 954


/Users/f006dwr/miniforge3/envs/nsi_fit/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [932]:
# At this stage, we assume we need to disagg all APTS & RES CONDO
pc_disagg = pc_one_bld_many[pc_one_bld_many['bld_type'].isin(['APTS', 'RES CONDO'])].copy()
 
# For all else, we assume there is only supposed to be one structure
# We'll only keep structures if they are very similar in size to 
# the largest structure (.95 or higher proportion of area) 
pc_poss_garag = pc_one_bld_many[~pc_one_bld_many['bld_type'].isin(['APTS', 'RES CONDO'])].copy()
pc_wo_garag = pc_poss_garag[pc_poss_garag['area_ratio'] >= .95].copy()
pc_wo_garag['n_bld'] = pc_wo_garag.groupby('parcel_number')['bfid'].transform('nunique').copy()
# Separate parcels with one building from more complex cases
# Add our simple cases to our processed dfs list
pc_one_bld_lowest_conf = pc_wo_garag[pc_wo_garag['n_bld'] == 1]

# For those parcels where all buildings are less than 100 m2, we
# will just keep the largest (these look like misplaced parcels
# that catch half of two separate houses from all of our checks)
pc_wo_g_remain = pc_wo_garag[pc_wo_garag['n_bld'] > 1].sort_values('m2_bld', ascending=False)
pc_wo_g_one = pc_wo_g_remain[(pc_wo_g_remain['area_ratio'] == 1) &
                             (pc_wo_g_remain['m2_bld'] < 100)].drop_duplicates('parcel_number', keep='first')
pc_one_bld_lowest_conf = pd.concat([pc_one_bld_lowest_conf, pc_wo_g_one], axis=0)
pc_bld_l.append(pc_one_bld_lowest_conf)
pc_wo_g_disagg = pc_wo_g_remain[~pc_wo_g_remain['parcel_number'].isin(pc_wo_g_one['parcel_number'])]

# Add these few to our processed dfs list
pc_bld_main = pd.concat(pc_bld_l, axis=0)

# Add rest to pc_disagg
pc_disagg = pd.concat([pc_disagg, pc_wo_g_disagg], axis=0)

print('Lowest confidence 1 to 1 matches: {}'.format(len(pc_one_bld_lowest_conf)))
print('Remaining 1 to many matches: {}'.format(len(pc_disagg['parcel_number'].unique())))
print('Remaining buildings: {}'.format(len(pc_disagg)))
print('Overall 1 to 1 matches identified: {}'.format(len(pc_bld_main)))

Lowest confidence 1 to 1 matches: 82
Remaining 1 to many matches: 156
Remaining buildings: 781
Overall 1 to 1 matches identified: 93955


We need to check if we are missing any tax records from `assess_linked` in our new datasets of tax records linked to one main building or tax records to disaggregate across structures. These can be missing because their building footprint is missing from `bld_fp`. Let's check out what's happening

In [ ]:
# These are records we can link to parcels but not to building footprints, 
# at least with the overlay
# dir_mat_missed = direct_matches[~(direct_matches['parcel_number'].isin(pc_bld_main['parcel_number'])) &
#                                 ~(direct_matches['parcel_number'].isin(pc_disagg['parcel_number']))]

# For example, this code will return an empty dataframe
# bld_fp_o[bld_fp_o['parcel_number'].isin(dir_mat_missed['parcel_number'])]

# But also can't find any of these records in the bld_fp data...
# dir_mat_missed[dir_mat_missed['PARCEL_ID'].isin(bld_fp['PARCEL_ID_NUM'])]
# dir_mat_missed[dir_mat_missed['ADDRESS'].isin(bld_fp['ADDRESS'])]

# Simply put, these are missing building footprints. See the following code for
# quick visualization of these instances
# Replace the BRT_ID with samples of BRT_ID from dir_mat_missed
# Some of these have structures but they're missing whereas others
# are vacant. I say we treat the Philly footprints as our baseline
# and treat these as examples of no building...
# We can use these parcel boundaries as a filter to remove NSI
# points inside of them as a sensitivity check

# from shapely.geometry import box
# import matplotlib.pyplot as plt
# temp = parcel[parcel['BRT_ID'] == '291124701']
# bbox = temp.total_bounds
# window = box(*bbox).buffer(.0001)

# fig, ax = plt.subplots()

# temp2 = bld_fp[bld_fp.geometry.intersects(window)]

# if not temp2.empty:
#     temp2.plot(ax=ax)
# temp.plot(ax=ax, color='none', edgecolor='red')


# Similarly, there are only 3 records below and they each seem to have an explanation for exclusion
# from further analysis. One is vacant according to recent satellite imagery. Another
# appears to have a missing building footprint in our data. Finally, one appears
# like it has incorrect links that even show up as problematic on the Properties web app

# indirect_matches[(~indirect_matches['parcel_number'].isin(pc_bld_main['parcel_number'])) &
#                   ~(indirect_matches['parcel_number'].isin(pc_disagg['parcel_number']))]

Now we want to aggregate tax records that represent buildings with many units and disaggregate tax records that represent parcels with many buildings. 

We check the parcels to disaggregate with the records we didn't link up to parcels. Some of these (maybe all) are the units in condos or apartment buildings and need to be aggregated with our parcels to disaggregate. If some of them don't link up, we have to check if we can link the tax record with one of our records in pc_bld_main, which suggests aggregating structure characteristics. Alternatively, we can see if we can link the tax record to a building footprint through a spatial join (through a parcel boundary and/or building footprint). Once we have no more stones unturned, we will have our set of records to disaggregate across structures. We also have to do aggregation in pc_bld_main for condos. 

In [933]:
# Only need value for aggregation and parcel_number for groupby
agg_cols = ['val_struct', 'BRT_ID']

# To do aggregation, there are different steps
# we have to take for parcels in pc_bld_main or pc_disagg
# based on whether they had a direct match to parcel or not
# For those with a direct match, we can just directly link BRT_ID
# that records in assess_no_link will get with a gpd sjoin to parcel
pc_dir_match = parcel[parcel['BRT_ID'].isin(direct_matches['BRT_ID'])]
dir_match_sp = gpd.sjoin(assess_no_link,
                         pc_dir_match,
                         predicate='within')

# But for those without a BRT_ID link, if they have a counterpart
# for aggregation we have to find that out through a spatial join
# We can remake the indirect_matches gdf and then drop
# the records in pc_bld_main and pc_disagg that are inside that
# Then we can merge on BRT_ID like we could for direct matches
indirect_matches = gpd.sjoin(assess_no_link,
                             parcel_no_match,
                             predicate='within')
# Then we want to remove any records whose parcel_number is already in
# pc_bld_main or pc_disagg to avoid double counting
ind_mask = ((~indirect_matches['parcel_number'].isin(pc_bld_main['parcel_number'])) |
             ~indirect_matches['parcel_number'].isin(pc_disagg['parcel_number']))
ind_match_sp = indirect_matches.loc[ind_mask].copy()

# Now create a geodataframe of these two 
assess_sp_link = pd.concat([dir_match_sp, ind_match_sp], axis=0)

assess_sp_link = assess_sp_link.loc[:, agg_cols].copy()

# Find parcel matches in pc_bld_main for aggregation
pc_agg_match = assess_sp_link[assess_sp_link['BRT_ID'].isin(pc_bld_main['parcel_number'])]
# Same for pc_disagg
pc_disagg_match = assess_sp_link[assess_sp_link['BRT_ID'].isin(pc_disagg['parcel_number'])]

# Get corresponding records from each of pc_agg_match & pc_disagg_match 
# so we can do aggregation (and subsequent disagg where needed)
pc_agg_add = pc_bld_main[pc_bld_main['parcel_number'].isin(pc_agg_match['BRT_ID'])].copy()
# Add relevant agg characteristics to dataframe
pc_agg_add = pc_agg_add.loc[:, agg_cols].copy()
# Then concat them
pc_agg_proc = pd.concat([pc_agg_match, pc_agg_add], axis=0)

# Repeat for pc_disagg_match
pc_disagg_add = pc_disagg[pc_disagg['parcel_number'].isin(pc_disagg_match['BRT_ID'])].copy()
pc_disagg_add = pc_disagg_add.loc[:, agg_cols].copy()
pc_disagg_proc = pd.concat([pc_disagg_match, pc_disagg_add], axis=0)

# Aggregate structure values, make dict, replace vals in main df
pc_agg_sum = pc_agg_proc.groupby('BRT_ID', as_index=False)['val_struct'].sum()
pc_agg_dict = dict(zip(pc_agg_sum['BRT_ID'], pc_agg_sum['val_struct']))
a_mask = pc_bld_main['parcel_number'].isin(pc_agg_sum['BRT_ID'])
pc_bld_main.loc[a_mask, 'val_struct'] = pc_bld_main.loc[a_mask, 'parcel_number'].map(pc_agg_dict)

pc_disagg_sum = pc_disagg_proc.groupby('BRT_ID', as_index=False)['val_struct'].sum()
pc_disagg_dict = dict(zip(pc_disagg_sum['BRT_ID'], pc_disagg_sum['val_struct']))
d_mask = pc_disagg['parcel_number'].isin(pc_disagg_sum['BRT_ID'])
pc_disagg.loc[d_mask, 'val_struct'] = pc_disagg.loc[d_mask, 'parcel_number'].map(pc_disagg_dict)

Drop structures with no value. Several checks suggest these are demolished structures/vacant lots. Some checks don't suggest this, but it's such a small number we can remove in our "best guess" inventory. 

Next, merge other structure characteristics into pc_bld_main and pc_disagg (RES1 and RES3 best guesses come later). Disaggregate for parcels with many structures and merge into pc_bld_main (need a new dataframe for these records). 

Finally, we want to check what tax records were lost along the way in this processing. Can we directly do tax record in building footprint spatial joins to recover? After leaving no stone unturned, we will call our residential inventory final and write it out with a parsimonious set of columns. 

In [961]:
inv_cols = ['parcel_number', 'basements', 'number_stories',
            'bld_type', 'bld_code_rest', 'val_struct', 
            'bfid', 'geometry']
assess_merge_cols = ['parcel_number', 'basements', 'number_stories',
                     'bld_type', 'bld_code_rest',
                     'building_code_description_new']

# Drop no val records
pc_bld_main_inv = pc_bld_main.loc[pc_bld_main['val_struct'] > 0].copy()
pc_disagg_inv = pc_disagg.loc[pc_disagg['val_struct'] > 0].copy()

no_val_main = len(pc_bld_main[~pc_bld_main['parcel_number'].isin(pc_bld_main_inv['parcel_number'])])
print('Dropped main records due to 0 value: {}'.format(no_val_main))
no_val_disagg = len(pc_disagg[~pc_disagg['parcel_number'].isin(pc_disagg_inv['parcel_number'])])
print('Dropped disagg records due to 0 value: {}'.format(no_val_disagg))

Dropped main records due to 0 value: 26
Dropped disagg records due to 0 value: 0


In [962]:
# Not sure why I did this just for disagg. This applies for 
# lots of bld_type, including those in pc_bld_main? 
# Shouldn't we have something like this for all structures?

# APTS, APT, and other types have different processing steps for getting
# a bld_type based estimate of number of stories
pc_disagg_inv['bld_str_len'] = pc_disagg_inv['bld_code_rest'].apply(lambda x: len(x))
non_apt_stry_mask = ((pc_disagg_inv['bld_type'] != 'APTS') &
                     (pc_disagg_inv['bld_str_len'] == 3))
pc_disagg_inv.loc[non_apt_stry_mask,
                  'stories_n'] = pc_disagg_inv['bld_code_rest'].apply(lambda x: x[0])
apt_w_stry_mask = (pc_disagg_inv['bld_type'] == 'APT')
pc_disagg_inv.loc[apt_w_stry_mask,
                  'stories_n'] = pc_disagg_inv['bld_code_rest'].apply(lambda x: x[2])

bld_code_new_stories = {'APTS - GARDEN': '2',
                        'APARTMENTS - LOW RISE': '2',
                        'APARTMENTS - BLT AS RESID': '3',
                        'APARTMENTS - HIGH RISE': '3',
                        'APARTMENTS - MID RISE': '3',
                        'APTS - HIGH RISE': '3'}
apt_no_stry_mask = (pc_disagg_inv['bld_type'] == 'APTS')
pc_disagg_inv.loc[apt_no_stry_mask,
                  'stories_n'] = pc_disagg_inv.loc[apt_no_stry_mask,
                                                   'building_code_description_new'].map(bld_code_new_stories)

# Get ratio of area to sum to assign values
pc_disagg_inv['total_m2'] = pc_disagg_inv.groupby('parcel_number')['m2_bld'].transform('sum')
pc_disagg_inv['m2_ratio'] = pc_disagg_inv['m2_bld'] / pc_disagg_inv['total_m2']
pc_disagg_inv['val_struct'] = pc_disagg_inv['val_struct'] * pc_disagg_inv['m2_ratio']

pc_disagg_inv = pc_disagg_inv.drop(columns=['total_m2', 'm2_ratio'])



`pc_bld_main_inv` has every parcel with a single building linked to all of its appropriate records. `pc_disagg_inv` has a record for each unique building for one parcel to many building records. The value is distributed across each of these structures. Now we have to concat these datasets, generate a unique id for each structure inventory record, and finish the inventory with the remaining characteristics. 

We'll keep: bld_type, parcel_number, bfid, val_struct, basements, number_stories, stories_n and drop the rest

After that, we'll assign RES1 & RES3 based on bld_type mappings and adjacent building processing

In [963]:
# Our main inventory
phil_inv = pd.concat([pc_bld_main_inv, pc_disagg_inv], axis=0)

In [1013]:
len(phil_inv)

94710

Now that we have our tax records uniquely linked/disaggregated to building foorprints, we can finalize our initialization of the inventory by getting the characteristics set up. This includes occupancy type code, structure value, stories, and basement type. We should have "best guess" versions of the columns as well as different alternatives based on defensible assumptions to test as sensitivity analyses. For example, we think some bld_type correspond to RES1 if they don't touch another structure but we can do a sensitivity check treating them as RES3 (if you think a row home should be RES3 no matter what). 

Now, we look at the subset of properties that have mixed RES1/RES3 *and* are the same size as row homes. We create a new column that indicates which of those touch at least one other building footprint. 

In [ ]:
touches_df = gpd.sjoin(gdf, gdf, how='inner', predicate='touches')
gdf['touches'] = gdf.index.isin(touches_df.index).astype(int)

## Process vulnerability

## Process reference data

## Hazard

In [ ]:
# HAZ_DIR_UZ
# HAZ_FILEN (need to modify with a wildcard for ensemble numbers from 01 to 50 - haz_nens)
# We need a function that turns any of the files into a depth grid
# We don't necessarily have to save these - might not want all that data
# I do want to do this for the "best estimate" though
# I only want the array part of the others
HAZ_CRS

In [ ]:
HAZ_DIR_UZ